In [1]:
!pip -q install grapheme

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# scripts.py

# maps language code → script name
LANG_TO_SCRIPT = {
    "hin": "Devanagari",
    "mar": "Devanagari",
    "san": "Devanagari",
    "nep": "Devanagari",
    "kok": "Devanagari",
    "mai": "Devanagari",
    "brx": "Devanagari",
    "doi": "Devanagari",
    "ben": "Bengali",
    "asm": "Bengali",     # Assamese uses Bengali script
    "tam": "Tamil",
    "tel": "Telugu",
    "kan": "Kannada",
    "mal": "Malayalam",
    "guj": "Gujarati",
    "pan": "Gurmukhi",
    "urd": "Arabic",
    "kas": "Arabic",      # Kashmiri often written in Arabic script
    "sid": "Ethiopic",    # Sidhi — verify this, might be Latin
    "mni": "Meitei",      # Manipuri
    "ori": "Odia",
}

# assign each script an integer index for the discriminator
ALL_SCRIPTS = sorted(set(LANG_TO_SCRIPT.values()))
SCRIPT_TO_IDX = {script: idx for idx, script in enumerate(ALL_SCRIPTS)}

NUM_SCRIPTS = len(ALL_SCRIPTS)

print(f"Scripts ({NUM_SCRIPTS}):")
for idx, script in enumerate(ALL_SCRIPTS):
    langs = [l for l, s in LANG_TO_SCRIPT.items() if s == script]
    print(f"  {idx}: {script:15s} ← {langs}")

Scripts (12):
  0: Arabic          ← ['urd', 'kas']
  1: Bengali         ← ['ben', 'asm']
  2: Devanagari      ← ['hin', 'mar', 'san', 'nep', 'kok', 'mai', 'brx', 'doi']
  3: Ethiopic        ← ['sid']
  4: Gujarati        ← ['guj']
  5: Gurmukhi        ← ['pan']
  6: Kannada         ← ['kan']
  7: Malayalam       ← ['mal']
  8: Meitei          ← ['mni']
  9: Odia            ← ['ori']
  10: Tamil           ← ['tam']
  11: Telugu          ← ['tel']


In [5]:
# dataset.py

import json
import grapheme
from torch.utils.data import Dataset
import torch
import random
from datasets import load_dataset, get_dataset_config_names

# Assuming LANG_TO_SCRIPT and SCRIPT_TO_IDX are imported or globally available from scripts.py
# This dependency should ideally be made explicit if dataset.py was a standalone file
# For Colab context, they are available because scripts.py is run first.

DIGRAPHS = [
    "tch", "sch",
    "sh", "ch", "th", "ph", "gh", "kh", "ng", "ck",
    "aa", "ee", "oo", "ai", "au", "ou", "oi",
    "dh", "bh", "jh",
]

def tokenize_roman(word):
    word = word.lower().strip()
    tokens = []
    i = 0
    while i < len(word):
        matched = False
        for digraph in DIGRAPHS:
            if word[i:i+len(digraph)] == digraph:
                tokens.append(digraph)
                i += len(digraph)
                matched = True
                break
        if not matched:
            if word[i] != ' ':
                tokens.append(word[i])
            i += 1
    return tokens

def tokenize_native(word):
    return list(grapheme.graphemes(word))


class TransliterationDataset(Dataset):

    def __init__(self, langs, split="train", max_per_lang=None):
        self.pairs = []   # now stores (native_word, roman_word, script_idx)

        # load vocab maps from disk
        with open("/content/drive/MyDrive/mlpr/native_vocab.json", encoding="utf-8") as f:
            self.native_vocab = json.load(f)
        with open("/content/drive/MyDrive/mlpr/roman_vocab.json", encoding="utf-8") as f:
            self.roman_vocab = json.load(f)

        # load all language datasets
        for lang in langs:
            # Ensure LANG_TO_SCRIPT and SCRIPT_TO_IDX are available in this scope
            # (they are from scripts.py cell)
            script     = LANG_TO_SCRIPT.get(lang, "Unknown")
            script_idx = SCRIPT_TO_IDX.get(script, 0)

            ds = load_dataset("eswardivi/Aksharantar", lang, split=split)
            count = 0
            for ex in ds:
                native = ex["native word"].strip()
                roman  = ex["english word"].strip()
                if native and roman:
                    self.pairs.append((native, roman, script_idx))  # ← added script_idx
                    count += 1
                if max_per_lang and count >= max_per_lang:
                    break        # cap this language, then move to next
            print(f"  {lang}: {count:,} pairs")

        print(f"total: {len(self.pairs):,} pairs from {len(langs)} languages")
        random.shuffle(self.pairs)

    def encode_native(self, word):
        """Convert a native word string to a list of vocab indices."""
        clusters = tokenize_native(word)
        return [
            self.native_vocab.get(c, self.native_vocab["<UNK>"])
            for c in clusters
        ]

    def encode_roman(self, word):
        """Convert a roman word string to a list of vocab indices."""
        tokens = tokenize_roman(word)
        offset = len(self.native_vocab)  # roman vocab starts after native vocab
        return [
            self.roman_vocab.get(t, self.roman_vocab["<UNK>"]) + offset
            for t in tokens
        ]

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        native_word, roman_word, script_idx = self.pairs[idx] # ← unpack 3 items
        # return as tensors
        return (
            torch.tensor(self.encode_native(native_word), dtype=torch.long),
            torch.tensor(self.encode_roman(roman_word),  dtype=torch.long),
            torch.tensor(script_idx,                      dtype=torch.long),
        )

In [6]:
import grapheme
from datasets import load_dataset, get_dataset_config_names
from collections import Counter
import json

# ---------- tokenizers ----------

DIGRAPHS = [
    "tch", "sch",
    "sh", "ch", "th", "ph", "gh", "kh", "ng", "ck",
    "aa", "ee", "oo", "ai", "au", "ou", "oi",
    "dh", "bh", "jh",
]

def tokenize_roman(word: str) -> list:
    word = word.lower().strip()
    tokens = []
    i = 0
    while i < len(word):
        matched = False
        for digraph in DIGRAPHS:
            if word[i:i+len(digraph)] == digraph:
                tokens.append(digraph)
                i += len(digraph)
                matched = True
                break
        if not matched:
            if word[i] != ' ':
                tokens.append(word[i])
            i += 1
    return tokens

def tokenize_native(word: str) -> list:
    return list(grapheme.graphemes(word))


# # ---------- build vocab ----------

# configs = get_dataset_config_names("eswardivi/Aksharantar")

# native_counter = Counter()
# roman_counter  = Counter()

# for lang in configs:
#     print(f"loading {lang}...")
#     ds = load_dataset("eswardivi/Aksharantar", lang, split="train")
#     for ex in ds:
#         for cluster in tokenize_native(ex["native word"]):
#             native_counter[cluster] += 1
#         for token in tokenize_roman(ex["english word"]):
#             roman_counter[token] += 1

# MIN_FREQ = 6

# SPECIAL = ["<PAD>", "<UNK>"]

# native_vocab = {tok: idx for idx, tok in enumerate(SPECIAL)}
# for tok, freq in native_counter.most_common():
#     if freq >= MIN_FREQ:
#         native_vocab[tok] = len(native_vocab)

# roman_vocab = {tok: idx for idx, tok in enumerate(SPECIAL)}
# for tok, freq in roman_counter.most_common():
#     if freq >= MIN_FREQ:
#         roman_vocab[tok] = len(roman_vocab)

# print(f"\nNative vocab size: {len(native_vocab)}")
# print(f"Roman vocab size:  {len(roman_vocab)}")

# # save to disk — you don't want to rebuild this every time
# # with open("native_vocab.json", "w", encoding="utf-8") as f:
# #     json.dump(native_vocab, f, ensure_ascii=False, indent=2)
# # with open("roman_vocab.json", "w", encoding="utf-8") as f:
# #     json.dump(roman_vocab, f, ensure_ascii=False, indent=2)

# # print("saved native_vocab.json and roman_vocab.json")

In [7]:
# model.py

import torch
import torch.nn as nn

class GraphemeEncoder(nn.Module):

    def __init__(self, vocab_size, embed_dim=128):
        super().__init__()
        # the embedding table: one vector per grapheme
        # vocab_size rows, embed_dim columns
        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=0,    # index 0 is <PAD>, its gradient is zeroed
        )
        # L2-normalize outputs to a unit sphere
        # this makes cosine similarity = dot product
        # which makes the loss numerically stable
        self.embed_dim = embed_dim

    def forward(self, token_ids):
        """
        token_ids: LongTensor of shape [batch_size, seq_len]
        returns:   FloatTensor of shape [batch_size, embed_dim]
        """
        # look up embedding for each token
        # shape: [batch_size, seq_len, embed_dim]
        embedded = self.embedding(token_ids)

        # average across the sequence dimension (dim=1)
        # shape: [batch_size, embed_dim]
        averaged = embedded.mean(dim=1)

        # L2 normalize: divide each vector by its own length
        # after this, every vector has length exactly 1.0
        # shape: [batch_size, embed_dim]  (unchanged)
        normalized = nn.functional.normalize(averaged, p=2, dim=1)

        return normalized


In [8]:
# adversarial.py

import torch
import torch.nn as nn

class GradientReversal(torch.autograd.Function):
    """
    Forward pass:  identity (passes input through unchanged)
    Backward pass: flips the sign of the gradient

    This is the trick that makes adversarial training work in one
    forward pass. The encoder sees a REVERSED gradient from the
    discriminator — so it moves in the direction that INCREASES
    the discriminator's loss (i.e. confuses it).
    """
    @staticmethod
    def forward(ctx, x, scale):
        ctx.scale = scale
        return x.clone()

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.scale * grad_output, None  # flip sign, no grad for scale


def grad_reverse(x, scale=1.0):
    return GradientReversal.apply(x, scale)


class ScriptDiscriminator(nn.Module):
    """
    Takes a phonetic vector z and tries to predict which script it came from.
    The encoder is trained to fool this — to produce vectors that look the
    same regardless of script.
    """
    def __init__(self, embed_dim, num_scripts):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_scripts)   # one logit per script
        )

    def forward(self, z, scale=1.0):
        # reverse gradient before passing to discriminator
        # encoder will be trained to maximize this loss (via sign flip)
        z_reversed = grad_reverse(z, scale)
        return self.net(z_reversed)

In [9]:
# train.py

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import json

# ── helpers ──────────────────────────────────────────────────

def collate_fn(batch):
    """
    batch is a list of (native_tensor, roman_tensor) pairs.
    tensors in each pair have different lengths — we pad them
    to the longest in the batch so they can stack into a matrix.
    """
    native_seqs, roman_seqs, script_ids = zip(*batch)

    native_padded = nn.utils.rnn.pad_sequence(
        native_seqs, batch_first=True, padding_value=0
    )
    roman_padded = nn.utils.rnn.pad_sequence(
        roman_seqs, batch_first=True, padding_value=0
    )
    script_ids = torch.stack(script_ids)   # shape [batch_size]

    return native_padded, roman_padded, script_ids


def infonce_loss(native_vecs, roman_vecs, temperature=0.07):
    """
    native_vecs: [batch_size, embed_dim]  — one vector per native word
    roman_vecs:  [batch_size, embed_dim]  — one vector per roman word

    For each native word i, its roman word i is the positive pair.
    All other roman words j≠i in the batch are negatives.

    Returns: scalar loss
    """
    # dot product of every native vec with every roman vec
    # shape: [batch_size, batch_size]
    # entry [i,j] = similarity between native word i and roman word j
    sim_matrix = torch.matmul(native_vecs, roman_vecs.T) / temperature

    # correct answer for row i is column i (the diagonal)
    labels = torch.arange(sim_matrix.size(0), device=sim_matrix.device)

    # cross entropy: for each row, treat col i as the "correct class"
    # this pushes sim[i,i] up and sim[i,j≠i] down
    loss = nn.functional.cross_entropy(sim_matrix, labels)
    return loss


# ── training loop ─────────────────────────────────────────────

def train():
    with open("/content/drive/MyDrive/mlpr/native_vocab.json", encoding="utf-8") as f:
        native_vocab = json.load(f)
    with open("/content/drive/MyDrive/mlpr/roman_vocab.json", encoding="utf-8") as f:
        roman_vocab = json.load(f)

    native_size      = len(native_vocab)
    total_vocab_size = native_size + len(roman_vocab)
    embed_dim        = 128

    encoder       = GraphemeEncoder(vocab_size=total_vocab_size, embed_dim=embed_dim)
    discriminator = ScriptDiscriminator(embed_dim=embed_dim, num_scripts=NUM_SCRIPTS)
    # load saved weights
    encoder.load_state_dict(torch.load("/content/drive/MyDrive/mlpr/encoder.pt"))
    discriminator.load_state_dict(torch.load("discriminator.pt"))

    # separate optimizers — they have competing goals
    enc_optimizer  = torch.optim.Adam(encoder.parameters(),       lr=3e-4)
    disc_optimizer = torch.optim.Adam(discriminator.parameters(), lr=3e-4)

    configs = get_dataset_config_names("eswardivi/Aksharantar")
    dataset = TransliterationDataset(configs, split="train", max_per_lang=50_000)
    loader  = DataLoader(dataset, batch_size=256, shuffle=True, collate_fn=collate_fn)

    # how much weight to give adversarial loss vs contrastive loss
    # start small — let contrastive loss dominate early
    lambda_adv = 0.5

    for epoch in range(10):
        total_contrastive = 0.0
        total_adversarial = 0.0

        for step, (native_batch, roman_batch, script_ids) in enumerate(loader):

            # ── Step 1: encode both sides ──────────────────────────────
            native_vecs = encoder(native_batch)   # [batch, 128]
            roman_vecs  = encoder(roman_batch)    # [batch, 128]

            # ── Step 2: contrastive loss ───────────────────────────────
            # pulls same-word vectors together across scripts
            loss_contrastive = infonce_loss(native_vecs, roman_vecs)

            # ── Step 3: adversarial loss ───────────────────────────────
            # discriminator tries to predict script from native vector
            # gradient reversal inside makes encoder try to CONFUSE it
            script_logits = discriminator(native_vecs, scale=lambda_adv)
            loss_adv      = nn.functional.cross_entropy(script_logits, script_ids)

            # ── Step 4: total loss and update ──────────────────────────
            # note: we do NOT multiply loss_adv by lambda here
            # because gradient reversal already scales it inside
            total_loss = loss_contrastive + loss_adv

            enc_optimizer.zero_grad()
            disc_optimizer.zero_grad()
            total_loss.backward()
            enc_optimizer.step()
            disc_optimizer.step()

            total_contrastive += loss_contrastive.item()
            total_adversarial += loss_adv.item()

            if step % 200 == 0:
                avg_c = total_contrastive / (step + 1)
                avg_a = total_adversarial / (step + 1)
                print(f"epoch {epoch}  step {step:5d}  "
                      f"contrastive {avg_c:.4f}  adversarial {avg_a:.4f}")

        print(f"── epoch {epoch} done  "
              f"contrastive {total_contrastive/len(loader):.4f}  "
              f"adversarial {total_adversarial/len(loader):.4f}")

    torch.save(encoder.state_dict(),       "encoder.pt")
    torch.save(discriminator.state_dict(), "discriminator.pt")
    print("saved encoder.pt and discriminator.pt")


In [10]:
import grapheme
# train()

In [11]:
# evaluate.py

import torch
import torch.nn.functional as F
import json

# ── load vocab ──────────────────────────────────────────────
with open("/content/drive/MyDrive/mlpr/native_vocab.json", encoding="utf-8") as f:
    native_vocab = json.load(f)
with open("/content/drive/MyDrive/mlpr/roman_vocab.json", encoding="utf-8") as f:
    roman_vocab = json.load(f)

native_size = len(native_vocab)
total_vocab  = native_size + len(roman_vocab)

idx2native = {v: k for k, v in native_vocab.items()}
idx2roman  = {v + native_size: k for k, v in roman_vocab.items()}

# ── load model ──────────────────────────────────────────────
model = GraphemeEncoder(vocab_size=total_vocab, embed_dim=128)
model.load_state_dict(torch.load("/content/drive/MyDrive/mlpr/encoder.pt", map_location="cpu"))
model.eval()

# extract the full embedding matrix
# shape: [total_vocab, 128]
all_embeddings = model.embedding.weight.detach()
all_embeddings = F.normalize(all_embeddings, p=2, dim=1)

# ── helper: nearest neighbours for one grapheme ─────────────
def nearest_native(grapheme, topk=10):
    """Given a native grapheme, find its closest native neighbours."""
    if grapheme not in native_vocab:
        print(f"'{grapheme}' not in vocab")
        return
    idx = native_vocab[grapheme]
    vec = all_embeddings[idx]                        # shape [128]

    # compute similarity to all native embeddings
    native_embeds = all_embeddings[:native_size]     # shape [native_size, 128]
    sims = native_embeds @ vec                       # shape [native_size]

    topk_indices = sims.topk(topk + 1).indices.tolist()

    print(f"\nNearest native neighbours to '{grapheme}':")
    for i in topk_indices:
        if i != idx:   # skip self
            print(f"  '{idx2native[i]}'  sim={sims[i].item():.3f}")

def nearest_roman(grapheme, topk=10):
    """Given a native grapheme, find its closest Roman neighbours."""
    if grapheme not in native_vocab:
        print(f"'{grapheme}' not in vocab")
        return
    idx = native_vocab[grapheme]
    vec = all_embeddings[idx]

    # roman embeddings start at native_size
    roman_embeds = all_embeddings[native_size:]      # shape [roman_size, 128]
    sims = roman_embeds @ vec                        # shape [roman_size]

    topk_indices = sims.topk(topk).indices.tolist()

    print(f"\nClosest Roman sounds to '{grapheme}':")
    for i in topk_indices:
        print(f"  '{idx2roman[i + native_size]}'  sim={sims[i].item():.3f}")

# ── run the checks ───────────────────────────────────────────

# Test 1: do equivalent 'k' sounds across scripts cluster together?
print("=" * 50)
print("TEST 1: cross-script clustering for 'k' sounds")
print("=" * 50)
# Test 1 extended — all k-sounds across all scripts
for grapheme in ['क', 'ক', 'க', 'ಕ', 'క', 'ക', 'ਕ', 'ક']:
    nearest_roman(grapheme, topk=3)

# Test 2: do similar-sounding native clusters find each other?
print("\n" + "=" * 50)
print("TEST 2: nearest native neighbours to Hindi 'क'")
print("=" * 50)
nearest_native('क', topk=10)

# Test 3: do clearly different sounds stay far apart?
print("\n" + "=" * 50)
print("TEST 3: similarity between 'क' (k) and 'न' (n)")
print("=" * 50)
idx_k = native_vocab.get('क')
idx_n = native_vocab.get('न')
if idx_k and idx_n:
    vec_k = all_embeddings[idx_k]
    vec_n = all_embeddings[idx_n]
    sim = (vec_k @ vec_n).item()
    print(f"  cosine similarity: {sim:.3f}")
    print(f"  (close to 1.0 = same sound, close to 0 or negative = different)")


TEST 1: cross-script clustering for 'k' sounds

Closest Roman sounds to 'क':
  'k'  sim=0.952
  'ck'  sim=0.904
  'q'  sim=0.843

Closest Roman sounds to 'ক':
  'k'  sim=0.926
  'ck'  sim=0.852
  'q'  sim=0.822

Closest Roman sounds to 'க':
  'k'  sim=0.637
  'g'  sim=0.606
  'q'  sim=0.588

Closest Roman sounds to 'ಕ':
  'k'  sim=0.795
  'ck'  sim=0.745
  'q'  sim=0.684

Closest Roman sounds to 'క':
  'k'  sim=0.850
  'ck'  sim=0.770
  'q'  sim=0.703

Closest Roman sounds to 'ക':
  'k'  sim=0.760
  'ck'  sim=0.736
  'q'  sim=0.721

Closest Roman sounds to 'ਕ':
  'q'  sim=0.919
  'k'  sim=0.893
  'ck'  sim=0.871

Closest Roman sounds to 'ક':
  'k'  sim=0.892
  'q'  sim=0.869
  'ck'  sim=0.835

TEST 2: nearest native neighbours to Hindi 'क'

Nearest native neighbours to 'क':
  'ক'  sim=0.917
  'ક'  sim=0.904
  'ਕ'  sim=0.901
  'का'  sim=0.897
  'କ'  sim=0.884
  'কা'  sim=0.881
  'ک'  sim=0.875
  'క'  sim=0.874
  'ಕ'  sim=0.845
  'क्'  sim=0.844

TEST 3: similarity between 'क' (k) and 'न

In [12]:
# scripts.py

# maps language code → script name
LANG_TO_SCRIPT = {
    "hin": "Devanagari",
    "mar": "Devanagari",
    "san": "Devanagari",
    "nep": "Devanagari",
    "kok": "Devanagari",
    "mai": "Devanagari",
    "brx": "Devanagari",
    "doi": "Devanagari",
    "ben": "Bengali",
    "asm": "Bengali",     # Assamese uses Bengali script
    "tam": "Tamil",
    "tel": "Telugu",
    "kan": "Kannada",
    "mal": "Malayalam",
    "guj": "Gujarati",
    "pan": "Gurmukhi",
    "urd": "Arabic",
    "kas": "Arabic",      # Kashmiri often written in Arabic script
    "sid": "Ethiopic",    # Sidhi — verify this, might be Latin
    "mni": "Meitei",      # Manipuri
    "ori": "Odia",
}

# assign each script an integer index for the discriminator
ALL_SCRIPTS = sorted(set(LANG_TO_SCRIPT.values()))
SCRIPT_TO_IDX = {script: idx for idx, script in enumerate(ALL_SCRIPTS)}

NUM_SCRIPTS = len(ALL_SCRIPTS)

print(f"Scripts ({NUM_SCRIPTS}):")
for idx, script in enumerate(ALL_SCRIPTS):
    langs = [l for l, s in LANG_TO_SCRIPT.items() if s == script]
    print(f"  {idx}: {script:15s} ← {langs}")

Scripts (12):
  0: Arabic          ← ['urd', 'kas']
  1: Bengali         ← ['ben', 'asm']
  2: Devanagari      ← ['hin', 'mar', 'san', 'nep', 'kok', 'mai', 'brx', 'doi']
  3: Ethiopic        ← ['sid']
  4: Gujarati        ← ['guj']
  5: Gurmukhi        ← ['pan']
  6: Kannada         ← ['kan']
  7: Malayalam       ← ['mal']
  8: Meitei          ← ['mni']
  9: Odia            ← ['ori']
  10: Tamil           ← ['tam']
  11: Telugu          ← ['tel']
